# GeoGram: Klasifikace sg/pl — IJP ÚJČ (všechny -ice obce)

Notebook spustí parser ÚJČ na všech 1 806 obcích s koncovkou `-ice` a uloží výsledky.

**Výstup:** `data/processed/ice_grammar_ujc.csv`

**Důležité:** Běh trvá ~30–45 minut (throttling 1 req/s, aby server ÚJČ nebyl přetížen).
Notebook ukládá průběžné výsledky do checkpointu — pokud se přeruší, lze ho spustit znovu a pokračuje tam, kde skončil.

## 1. Setup

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from geogram.ujc import fetch_ujc_number, SINGULAR, PLURAL, BOTH, UNKNOWN

PROCESSED_DIR = project_root / 'data' / 'processed'
INPUT_CSV     = PROCESSED_DIR / 'municipalities_ice_integrated.csv'
CHECKPOINT    = PROCESSED_DIR / 'ujc_checkpoint.csv'
OUTPUT_CSV    = PROCESSED_DIR / 'ice_grammar_ujc.csv'

SLEEP_BETWEEN_REQUESTS = 1.0  # sekundy — buď ohleduplný k serveru ÚJČ
CHECKPOINT_EVERY = 50         # uložit checkpoint každých N obcí

print(f'Input:      {INPUT_CSV}')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Output:     {OUTPUT_CSV}')

## 2. Načtení dat a resumování checkpointu

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Načteno {len(df):,} obcí z {INPUT_CSV.name}')

# Resumování — načíst hotové výsledky z checkpointu
if CHECKPOINT.exists():
    done = pd.read_csv(CHECKPOINT)
    done_codes = set(done['code'].tolist())
    print(f'Checkpoint nalezen: {len(done):,} obcí již zpracováno, přeskočím je.')
else:
    done = pd.DataFrame(columns=['code', 'name', 'ujc_number'])
    done_codes = set()
    print('Žádný checkpoint — začínám od začátku.')

todo = df[~df['code'].isin(done_codes)].reset_index(drop=True)
print(f'Zbývá zpracovat: {len(todo):,} obcí')

## 3. Klasifikace (s throttlingem a checkpointováním)

In [ ]:
results = done.to_dict('records')  # začít s již hotovými
errors = []

pbar = tqdm(todo.iterrows(), total=len(todo), unit='obec',
            desc='ÚJČ klasifikace', dynamic_ncols=True)

for i, row in pbar:
    name = row['name']
    code = row['code']

    try:
        number = fetch_ujc_number(name)
    except Exception as exc:
        number = 'error'
        errors.append({'code': code, 'name': name, 'error': str(exc)})

    results.append({'code': code, 'name': name, 'ujc_number': number})

    pbar.set_postfix({'poslední': name, 'výsledek': number, 'chyby': len(errors)})

    # Checkpoint
    pos = i + 1
    if pos % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False, encoding='utf-8')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

pbar.close()
print(f'\nHotovo. Chyby při síťovém volání: {len(errors)}')
if errors:
    print('Chybné obce:')
    for e in errors:
        print(f"  {e['name']}: {e['error']}")

## 4. Uložení výsledků

In [ ]:
df_results = pd.DataFrame(results)

# Spojit s původními daty (demografie, region atd.)
df_final = df.merge(df_results[['code', 'ujc_number']], on='code', how='left')

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
# Smazat checkpoint — výsledky jsou uloženy v output
if CHECKPOINT.exists():
    CHECKPOINT.unlink()

print(f'Uloženo {len(df_final):,} řádků do {OUTPUT_CSV}')
df_final[['name', 'region_name', 'population_total', 'ujc_number']].head(10)

## 5. Základní statistiky

In [ ]:
counts = df_final['ujc_number'].value_counts()
total = len(df_final)

print('=== Distribuce gramatického čísla (ÚJČ) ===')
for label, n in counts.items():
    print(f'  {label:<12}: {n:>5}  ({n/total*100:.1f} %)')

print(f'\nCelkem obcí: {total:,}')

In [ ]:
# Rozložení sg/pl po krajích
df_known = df_final[df_final['ujc_number'].isin([SINGULAR, PLURAL, BOTH])]

pivot = (
    df_known
    .groupby(['region_name', 'ujc_number'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values('total', ascending=False)
)

print('=== Sg/pl po krajích ===')
print(pivot.to_string())

In [ ]:
# Příklady z každé kategorie (top 5 podle populace)
for label in [SINGULAR, PLURAL, BOTH, UNKNOWN, 'error']:
    subset = df_final[df_final['ujc_number'] == label]
    if subset.empty:
        continue
    top = subset.nlargest(5, 'population_total')[['name', 'region_name', 'population_total']]
    print(f'\n--- {label.upper()} (top 5 podle populace) ---')
    print(top.to_string(index=False))